#  Fraud Detection Pipeline
## Phát hiện giao dịch gian lận trong thương mại điện tử

### Mô tả bài toán
- **Dataset**: 300,000 giao dịch thương mại điện tử
- **Mục tiêu**: Phân loại giao dịch gian lận (`Is Fraudulent = 1`)
- **Thách thức**: Dataset mất cân bằng nghiêm trọng (~5% fraud)

### Metric đánh giá chính
| Metric | Lý do chọn |
|--------|-----------|
| **F2-score** | Metric chính — penalize việc bỏ sót fraud nặng hơn false alarm |
| **PR-AUC** | Đánh giá tổng thể khả năng phân biệt fraud trên mọi threshold |
| **Recall** | Tỷ lệ fraud thực sự bị phát hiện |


### Pipeline tổng quan
```
Load Data → EDA → Feature Engineering → Train/Test Split
→ Group Statistics → Pattern Features
→ Spark MLlib: LR + RF + GBT → Chọn Best Model (F2)
→ Threshold Tuning (F2) → Final Evaluation
```

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    average_precision_score, precision_recall_curve,
    roc_auc_score, f1_score, fbeta_score,
    precision_score, recall_score
)

from pyspark.sql import SparkSession
from pyspark.ml import Pipeline as SparkPipeline
from pyspark.ml.feature import (
    VectorAssembler, StandardScaler as SparkScaler,
    StringIndexer, OneHotEncoder as SparkOHE
)
from pyspark.ml.classification import (
    LogisticRegression as SparkLR,
    RandomForestClassifier as SparkRF,
    GBTClassifier
)
from pyspark.sql.functions import col, when

TARGET_COL = "Is Fraudulent"
DATE_COL   = "Transaction Date"

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except Exception:
    plt.style.use("default")

print(" Imports loaded")

## Load & Khám phá dữ liệu (EDA)

### Các vấn đề phát hiện từ EDA
1. **Mixed datetime format** → dùng `format='mixed'`
2. **High-cardinality features** (`IP Address`, `Customer Location`) → leakage nghiêm trọng (train corr=0.99, test corr=0.006) → **DROP hoàn toàn**
3. **Customer Age** có giá trị < 18 → clean bằng median của train


In [ ]:
HDFS_PATH = "hdfs://localhost:9000/final/fraud_300k_raw_temporal.csv"

spark = SparkSession.builder \
    .appName("FraudDetection_HDFS_MLlib") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.sql.execution.pyspark.udf.faulthandler.enabled", "true") \
    .config("spark.python.worker.faulthandler.enabled", "true") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print(f" Spark {spark.version} | Master: {spark.sparkContext.master}")

df_spark_raw = (
    spark.read
         .option("header", "true")
         .option("inferSchema", "false")
         .option("multiLine", "true")
         .option("mode", "PERMISSIVE")
         .option("quote", '"')
         .option("escape", '"')
         .csv(HDFS_PATH)
)

df_spark_raw = df_spark_raw.toDF(*[c.strip() for c in df_spark_raw.columns])

spark_rows = df_spark_raw.count()
print(f" Spark read from HDFS: {HDFS_PATH}")
print(f" Spark DataFrame: {spark_rows:,} rows × {len(df_spark_raw.columns)} columns")
df_spark_raw.printSchema()

if spark_rows == 0:
    raise ValueError("Spark đọc được 0 dòng từ HDFS. Kiểm tra lại HDFS_PATH hoặc file CSV trên HDFS.")

print("\n Check label values from Spark:")
df_spark_raw.select(TARGET_COL).distinct().show(30, truncate=False)

df = df_spark_raw.toPandas()
df.columns = [c.strip() for c in df.columns]

required_cols = [DATE_COL, TARGET_COL]
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise KeyError(f"Thiếu cột bắt buộc: {missing_cols}. Các cột hiện có: {df.columns.tolist()}")

date_raw = df[DATE_COL].copy()
try:
    df[DATE_COL] = pd.to_datetime(date_raw, format="mixed", dayfirst=False, errors="coerce")
except Exception:
    df[DATE_COL] = pd.to_datetime(date_raw, dayfirst=False, errors="coerce")

if df[DATE_COL].notna().sum() == 0:
    try:
        date_alt = pd.to_datetime(date_raw, format="mixed", dayfirst=True, errors="coerce")
    except Exception:
        date_alt = pd.to_datetime(date_raw, dayfirst=True, errors="coerce")
    if date_alt.notna().sum() > df[DATE_COL].notna().sum():
        df[DATE_COL] = date_alt

print("\n Giá trị gốc của label trước xử lý:")
print(df[TARGET_COL].value_counts(dropna=False).head(20))

label_raw = df[TARGET_COL]
if label_raw.dtype == "bool":
    df[TARGET_COL] = label_raw.astype(int)
else:
    label_str = (
        label_raw.astype(str)
                 .str.strip()
                 .str.replace('"', '', regex=False)
                 .str.replace("'", "", regex=False)
                 .str.lower()
    )
    label_map = {
        "1": 1, "1.0": 1, "true": 1, "t": 1, "yes": 1, "y": 1,
        "fraud": 1, "fraudulent": 1, "is fraud": 1, "is fraudulent": 1,
        "0": 0, "0.0": 0, "false": 0, "f": 0, "no": 0, "n": 0,
        "non-fraud": 0, "non fraud": 0, "not fraud": 0,
        "not fraudulent": 0, "legitimate": 0, "normal": 0
    }
    mapped_label = label_str.map(label_map)
    numeric_label = pd.to_numeric(label_raw, errors="coerce")
    df[TARGET_COL] = mapped_label.fillna(numeric_label)

numeric_cols = [
    "Transaction Amount", "Quantity", "Customer Age",
    "Account Age Days", "Transaction Hour"
]
for c in numeric_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

before_rows = len(df)
df = df.dropna(subset=[TARGET_COL]).copy()
df[TARGET_COL] = df[TARGET_COL].astype(int)
df = df[df[TARGET_COL].isin([0, 1])].copy()
after_rows = len(df)

print("\n Giá trị label sau xử lý:")
print(df[TARGET_COL].value_counts(dropna=False).sort_index())
print(f"\n Rows before label cleaning: {before_rows:,}")
print(f" Rows after label cleaning : {after_rows:,}")

if after_rows == 0:
    raise ValueError("Sau khi xử lý label, df còn 0 dòng. Kiểm tra lại TARGET_COL hoặc cách đọc CSV từ HDFS.")
if df[TARGET_COL].nunique() < 2:
    raise ValueError(f"Cột {TARGET_COL} chỉ còn một lớp: {df[TARGET_COL].unique()}. Stratified split cần đủ 0 và 1.")

fraud_rate = df[TARGET_COL].mean()
imbalance_text = f"imbalance ratio 1:{int((1 - fraud_rate) / fraud_rate)}" if fraud_rate > 0 else "imbalance ratio không tính được"

if df[DATE_COL].notna().any():
    date_text = f"{df[DATE_COL].min().date()} → {df[DATE_COL].max().date()}"
else:
    date_text = "Không parse được ngày hợp lệ"

print(f"\n Loaded : {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f" Range  : {date_text}")
print(f"⚖  Fraud  : {fraud_rate*100:.2f}%  ({imbalance_text})")


fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("EDA — Key Fraud Patterns", fontsize=13, fontweight="bold")

# 1. Class imbalance
counts = df[TARGET_COL].value_counts().reindex([0, 1], fill_value=0)
axes[0].bar(["Non-Fraud", "Fraud"], counts.values,
            color=["#4C9BE8", "#E84C4C"], edgecolor="white")
axes[0].set_title("Class Distribution")
for i, v in enumerate(counts.values):
    axes[0].text(i, v + max(counts.values) * 0.01,
                 f"{v:,}\n({v/len(df)*100:.1f}%)",
                 ha="center", fontsize=9)

hour_fraud = df.groupby("Transaction Hour")[TARGET_COL].mean() * 100
axes[1].bar(hour_fraud.index, hour_fraud.values,
            color=["#E84C4C" if h <= 5 else "#4C9BE8" for h in hour_fraud.index],
            alpha=0.8)
axes[1].set_title("Fraud Rate by Hour\n(Red = Hour 0–5)")
axes[1].set_xlabel("Hour")
axes[1].set_ylabel("Fraud Rate (%)")
axes[1].axhline(fraud_rate * 100, color="gray", linestyle="--", label="Baseline")
axes[1].legend()

df_tmp = df.copy()
df_tmp["age_bucket"] = pd.cut(
    df_tmp["Account Age Days"],
    bins=[0, 30, 60, 90, 180, 365, 999],
    labels=["0-30", "31-60", "61-90", "91-180", "181-365", "365+"],
    include_lowest=True
)
age_fraud = df_tmp.groupby("age_bucket")[TARGET_COL].mean() * 100
axes[2].bar(age_fraud.index, age_fraud.values,
            color=["#E84C4C" if i == 0 else "#4C9BE8" for i in range(len(age_fraud))],
            alpha=0.8)
axes[2].set_title("Fraud Rate by Account Age\n(Red = 0–30 days)")
axes[2].set_ylabel("Fraud Rate (%)")
axes[2].axhline(fraud_rate * 100, color="gray", linestyle="--")

plt.tight_layout()
plt.show()